In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
################################ Data generation ns   ##################################
# This code is adapted from the FNO's project
# https://github.com/zongyi-li/fourier_neural_operator
# Author: Wuzhe Xu
# Date: 07/27/2022
########################################################################################

import torch
import math
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import cm

# from drawnow import drawnow, figure

from timeit import default_timer

import scipy.io
from tqdm import tqdm

In [4]:
####################################################################
#w0: initial vorticity
#f: forcing term
#visc: viscosity (1/Re)
#T: final time
#delta_t: internal time-step for solve (descrease if blow-up)
#record_steps: number of in-time snapshots to record
def navier_stokes_2d(w0, f, visc, T, delta_t, record_steps):

    #Grid size - must be power of 2
    N = w0.size()[-1]

    #Maximum frequency 
    k_max = math.floor(N/2.0)

    #Number of steps to final time
    total_steps = math.ceil(T/delta_t)

    #Initial vorticity to Fourier space
    w_fourier = torch.rfft(w0, 2, normalized=False, onesided=False)

    #Forcing to Fourier space
    f_fourier = torch.rfft(f, 2, normalized=False, onesided=False)

    #If same forcing for the whole batch
    if len(f_fourier.size()) < len(w_fourier.size()):
        f_fourier = torch.unsqueeze(f_fourier, 0)

    #Record solution every this number of steps
    record_time = math.floor(total_steps/record_steps)

    #Wavenumbers in y-direction
    k_y = torch.cat((torch.arange(start=0, end=k_max, step=1, device=w0.device), torch.arange(start=-k_max, end=0, step=1, device=w0.device)), 0).repeat(N,1)
    #Wavenumbers in x-direction
    k_x = k_y.transpose(0,1)
    #Negative Laplacian in Fourier space
    neg_laplacian_fourier = 4*(math.pi**2)*(k_x**2 + k_y**2)
    neg_laplacian_fourier[0,0] = 1.0
    #Dealiasing mask
    dealias = torch.unsqueeze(torch.logical_and(torch.abs(k_y) <= (2.0/3.0)*k_max, torch.abs(k_x) <= (2.0/3.0)*k_max).float(), 0)

    #Initial velocity field
    psi_fourier = w_fourier.clone()
    psi_fourier[...,0] = psi_fourier[...,0]/neg_laplacian_fourier
    psi_fourier[...,1] = psi_fourier[...,1]/neg_laplacian_fourier

    #Velocity field in x-direction = psi_y
    u = psi_fourier.clone()
    temp = u[...,0].clone()
    u[...,0] = -2*math.pi*k_y*u[...,1]
    u[...,1] = 2*math.pi*k_y*temp
    # velocity x 0 = (d(psi)/d(y))
    psi_y_0 = torch.irfft(u, 2, normalized=False, onesided=False, signal_sizes=(N,N))

    #Velocity field in y-direction = -psi_x
    v = psi_fourier.clone()
    temp = v[...,0].clone()
    v[...,0] = 2*math.pi*k_x*v[...,1]
    v[...,1] = -2*math.pi*k_x*temp
    # velocity y 0 = (-d(psi)/d(x))
    psi_x_0 = -torch.irfft(v, 2, normalized=False, onesided=False, signal_sizes=(N,N))

    #Saving solution and time
    vorticity_through_time = torch.zeros(*w0.size(), record_steps, device=w0.device)
    psi_x_through_time = torch.zeros(*w0.size(), record_steps, device=w0.device)
    psi_y_through_time = torch.zeros(*w0.size(), record_steps, device=w0.device)
    time_when_record = torch.zeros(record_steps, device=w0.device)

    #Record counter
    c = 0
    #Physical time
    t = 0.0
    for j in tqdm(range(total_steps), desc="Steps time"):
        #Stream function in Fourier space: solve Poisson equation
        #print('total steps=', steps, ' current step=', j)
        psi_fourier = w_fourier.clone()
        psi_fourier[...,0] = psi_fourier[...,0]/neg_laplacian_fourier
        psi_fourier[...,1] = psi_fourier[...,1]/neg_laplacian_fourier

        #Velocity field in x-direction = psi_y
        u = psi_fourier.clone()
        temp = u[...,0].clone()
        u[...,0] = -2*math.pi*k_y*u[...,1]
        u[...,1] = 2*math.pi*k_y*temp
        # velocity x (d(psi)/d(y))
        psi_y = torch.irfft(u, 2, normalized=False, onesided=False, signal_sizes=(N,N))

        #Velocity field in y-direction = -psi_x
        v = psi_fourier.clone()
        temp = v[...,0].clone()
        v[...,0] = 2*math.pi*k_x*v[...,1]
        v[...,1] = -2*math.pi*k_x*temp
        # velocity y (-d(psi)/d(x))
        psi_x = torch.irfft(v, 2, normalized=False, onesided=False, signal_sizes=(N,N))

        #Partial x of vorticity
        w_x = w_fourier.clone()
        temp = w_x[...,0].clone()
        w_x[...,0] = -2*math.pi*k_x*w_x[...,1]
        w_x[...,1] = 2*math.pi*k_x*temp
        # (d(w)/dx)
        w_x = torch.irfft(w_x, 2, normalized=False, onesided=False, signal_sizes=(N,N))

        #Partial y of vorticity
        w_y = w_fourier.clone()
        temp = w_y[...,0].clone()
        w_y[...,0] = -2*math.pi*k_y*w_y[...,1]
        w_y[...,1] = 2*math.pi*k_y*temp
        # (d(w)/dy)
        w_y = torch.irfft(w_y, 2, normalized=False, onesided=False, signal_sizes=(N,N))

        #Non-linear term (u.grad(w)): compute in physical space then back to Fourier space
        nonlinear_term = torch.rfft(psi_y*w_x + psi_x*w_y, 2, normalized=False, onesided=False)

        #Dealias
        nonlinear_term[...,0] = dealias* nonlinear_term[...,0]
        nonlinear_term[...,1] = dealias* nonlinear_term[...,1]

        #Cranck-Nicholson update
        w_fourier[...,0] = (-delta_t*nonlinear_term[...,0] + delta_t*f_fourier[...,0] + (1.0 - 0.5*delta_t*visc*neg_laplacian_fourier)*w_fourier[...,0])/(1.0 + 0.5*delta_t*visc*neg_laplacian_fourier)
        w_fourier[...,1] = (-delta_t*nonlinear_term[...,1] + delta_t*f_fourier[...,1] + (1.0 - 0.5*delta_t*visc*neg_laplacian_fourier)*w_fourier[...,1])/(1.0 + 0.5*delta_t*visc*neg_laplacian_fourier)

        #Update real time (used only for recording)
        t += delta_t

        if (j+1) % record_time == 0:
            #Solution in physical space
            w = torch.irfft(w_fourier, 2, normalized=False, onesided=False, signal_sizes=(N,N))

            psi_fourier = w_fourier.clone()
            psi_fourier[...,0] = psi_fourier[...,0]/neg_laplacian_fourier
            psi_fourier[...,1] = psi_fourier[...,1]/neg_laplacian_fourier

            #Velocity field in x-direction = psi_y
            u = psi_fourier.clone()
            temp = u[...,0].clone()
            u[...,0] = -2*math.pi*k_y*u[...,1]
            u[...,1] = 2*math.pi*k_y*temp
            # velocity x (d(psi)/d(y))
            psi_y = torch.irfft(u, 2, normalized=False, onesided=False, signal_sizes=(N,N))

            #Velocity field in y-direction = -psi_x
            v = psi_fourier.clone()
            temp = v[...,0].clone()
            v[...,0] = 2*math.pi*k_x*v[...,1]
            v[...,1] = -2*math.pi*k_x*temp
            # velocity y (-d(psi)/d(x))
            psi_x = torch.irfft(v, 2, normalized=False, onesided=False, signal_sizes=(N,N))

            #Record solution and time
            vorticity_through_time[...,c] = w
            psi_x_through_time[...,c] = -psi_x
            psi_y_through_time[...,c] = psi_y
            time_when_record[c] = t

            c += 1


    return psi_x_0, psi_y_0, vorticity_through_time, psi_x_through_time, psi_y_through_time, time_when_record


def taylor_green_ic(batch, Nx, amp_range=(1.0, 1.2), add_noise=0.25):
    t  = torch.linspace(0, 1, Nx+1)[:-1]
    X, Y = torch.meshgrid(t, t)

    amplitude = torch.empty(batch, 1, 1).uniform_(*amp_range)
    phase_x = torch.rand(batch, 1, 1)
    phase_y = torch.rand(batch, 1, 1)

    initial_vorticity_field = 2*math.pi * amplitude * torch.sin(2*math.pi*(X+phase_x)) * torch.sin(2*math.pi*(Y+phase_y))

    if add_noise > 0:
        random_noise = add_noise * torch.randn_like(initial_vorticity_field)
        initial_vorticity_field += random_noise - random_noise.mean(dim=(-2,-1), keepdim=True)

    return initial_vorticity_field 


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

### parameters
nu = 1.0e-4
#Resolution
size = 64

Nx = size

dt = 1e-3
dt_record_every = 0.02 # 
T_train = 15
T_test = 25

#Number of snapshots from solution
record_steps_train = int(T_train/dt_record_every)
record_steps_test = int(T_test/dt_record_every)

#Number of steps to take in data (usually = record_steps)
steps_taking_train = record_steps_train
steps_taking_test = record_steps_test

#Number of solutions to generate
Number_of_train_data = 200
Number_of_test_data = 50
size_of_train_col = Number_of_train_data*steps_taking_train


#Forcing function
t = torch.linspace(0, 1, size+1, device=device)
t = t[0:-1]
x = t


X, Y = torch.meshgrid(t, t)
# f = nu*(2*np.pi)**2*(torch.sin(2*math.pi*(X + Y)) + torch.cos(2*math.pi*(X + Y)))

# f = 0.1*(torch.sin(2*math.pi*(X + Y)) + torch.cos(2*math.pi*(X + Y)))
f = torch.zeros_like(X)

f = f.to(device)

#Batch size
bsize = 10

####################################################
c = 0
t0 =default_timer()
#Inputs
vorticity_init = torch.zeros(Number_of_train_data, size, size)
psi_x_init = torch.zeros(Number_of_train_data, size, size)
psi_y_init = torch.zeros(Number_of_train_data, size, size)
#Solutions
vorticity = torch.zeros(Number_of_train_data, size, size, record_steps_train)
psi_x = torch.zeros(Number_of_train_data, size, size, record_steps_train)
psi_y = torch.zeros(Number_of_train_data, size, size, record_steps_train)

time_when_record_train = torch.zeros(Number_of_train_data, record_steps_train) 

for j in range(Number_of_train_data//bsize):

    #Sample random feilds
    w0 = taylor_green_ic(bsize, Nx)
    w0 = w0.to(device)
    #Solve NS

    with torch.no_grad():
        psi_x_0, psi_y_0, vorticity_through_time, psi_x_through_time, psi_y_through_time, time_when_record = navier_stokes_2d(w0, f, nu, T_train, dt, record_steps_train)
    # vorticity y(0)
    vorticity_init[c:(c+bsize),...] = w0.cpu()
    # velocity y(0) = (-d(psi(0))/d(x))
    psi_x_init[c:(c+bsize),...] = psi_x_0.cpu()
    # velocity x(0) = (d(psi(0))/d(y))
    psi_y_init[c:(c+bsize),...] = psi_y_0.cpu()
    # vorticity y(t)
    vorticity[c:(c+bsize),...] = vorticity_through_time.cpu()
    # velocity y(t) = (-d(psi(t))/d(x))
    psi_x[c:(c+bsize),...] = psi_x_through_time.cpu()
    # velocity x(t) = (d(psi(t))/d(y))
    psi_y[c:(c+bsize),...] = psi_y_through_time.cpu()

    tr = time_when_record.cpu().unsqueeze(0).repeat(bsize, 1)
    time_when_record_train[c:c+bsize] = tr

    del w0, psi_x_0, psi_y_0, vorticity_through_time, psi_x_through_time, psi_y_through_time, time_when_record
    torch.cuda.empty_cache()

    c += bsize
    t1 = default_timer()
    print(j, c, t1-t0)


train_initial_vorticity = vorticity_init.cpu().detach().numpy()
train_initial_psi_x = psi_x_init.detach().numpy()
train_initial_psi_y = psi_y_init.detach().numpy()

train_vorticity_evolution = vorticity.cpu().detach().numpy()
train_psi_x_evolution = psi_x.detach().numpy()
train_psi_y_evolution = psi_y.detach().numpy()



############################################

####################################################
c = 0
t0 =default_timer()
#Inputs
test_vorticity_init = torch.zeros(Number_of_test_data, size, size)
psi_x_init_test = torch.zeros(Number_of_test_data, size, size)
psi_y_init_test = torch.zeros(Number_of_test_data, size, size)
#Solutions
vorticity_test = torch.zeros(Number_of_test_data, size, size, record_steps_test)

time_when_record_test = torch.zeros(Number_of_test_data, record_steps_test) 


for j in range(Number_of_test_data//bsize):

    #Sample random feilds
    w0 = taylor_green_ic(bsize, Nx)
    w0 = w0.to(device)
    #Solve NS
    with torch.no_grad():
        psi_x_0, psi_y_0, vorticity_through_time, psi_x_through_time, psi_y_through_time, time_when_record = navier_stokes_2d(w0, f, nu, T_test, dt, record_steps_test)

    test_vorticity_init[c:(c+bsize),...] = w0.cpu()
    psi_x_init_test[c:(c+bsize),...] = psi_x_0.cpu()
    psi_y_init_test[c:(c+bsize),...] = psi_y_0.cpu()
    vorticity_test[c:(c+bsize),...] = vorticity_through_time.cpu()

    tr = time_when_record.cpu().unsqueeze(0).repeat(bsize, 1)
    time_when_record_test[c:c+bsize] = tr

    del w0, psi_x_0, psi_y_0, vorticity_through_time, psi_x_through_time, psi_y_through_time, time_when_record
    torch.cuda.empty_cache()

    c += bsize
    t1 = default_timer()
    print(j, c, t1-t0)
    

test_initial_vorticity = test_vorticity_init.cpu().detach().numpy()
test_initial_psi_x = psi_x_init_test.detach().numpy()
test_initial_psi_y = psi_y_init_test.detach().numpy()
test_vorticity_evolution = vorticity_test.cpu().detach().numpy()

################# previous code copied from FNO project #################
########## save reference for DON ########################

train_branch_matrix = np.zeros((size_of_train_col, Nx**2))
train_psi_x_matrix = np.zeros((size_of_train_col, Nx**2))
train_psi_y_matrix = np.zeros((size_of_train_col, Nx**2))

for i in range(Number_of_train_data):
    for j in range(steps_taking_train):
        tmp = train_vorticity_evolution[i, :, :, j]
        tmp_psi_x = train_psi_x_evolution[i, :, :, j]
        tmp_psi_y = train_psi_y_evolution[i, :, :, j]
        train_branch_matrix[i*steps_taking_train+j, :] = tmp.reshape(1, Nx**2)
        train_psi_x_matrix[i*steps_taking_train+j, :] = tmp_psi_x.reshape(1, Nx**2)
        train_psi_y_matrix[i*steps_taking_train+j, :] = tmp_psi_y.reshape(1, Nx**2)

test_branch_matrix = test_initial_vorticity.reshape(Number_of_test_data, Nx**2)
test_psi_x_matrix = test_initial_psi_x.reshape(Number_of_test_data, Nx**2)
test_psi_y_matrix = test_initial_psi_y.reshape(Number_of_test_data, Nx**2)

test_evolution_matrix = np.zeros((Number_of_test_data, steps_taking_test, Nx**2))

for i in range(Number_of_test_data):
    for j in range(steps_taking_test):
        test_evolution_matrix[i,j, :] = test_vorticity_evolution[i, :, :, j].reshape(1, Nx**2)

test_branch_input = test_branch_matrix

####################################################################
# save data

filename = 'DON' + \
           '_nu_' + str(nu) + \
           '_size_' + str(size) + \
           '_dtRecord_' + str(dt_record_every) + \
           '_dtCycle_' + str(dt) + \
            '_trainTime_' + str(T_train) + \
            '_testTime_' + str(T_test) + \
           '_NumTrain_' + str(Number_of_train_data) + \
           '_NumTest_' + str(Number_of_test_data)

subfolder = f"{filename}"


data_root = "Data"
out_dir = os.path.join(data_root, subfolder)
os.makedirs(out_dir, exist_ok=True)

npy_path = os.path.join(out_dir, f"{filename}.npy")

with open(npy_path, 'wb') as ss:
    np.save(ss, x.cpu().numpy())
    np.save(ss, train_branch_matrix)
    np.save(ss, train_psi_x_matrix)
    np.save(ss, train_psi_y_matrix)

    np.save(ss, test_branch_input)
    np.save(ss, test_branch_matrix)
    np.save(ss, test_psi_x_matrix)
    np.save(ss, test_psi_y_matrix)
    np.save(ss, test_evolution_matrix)


time_vector_train = time_when_record_train[0].cpu().numpy()  
time_vector_test  = time_when_record_test [0].cpu().numpy() 

train_mat_path = os.path.join(out_dir, "ns_data_fno_train.mat")

scipy.io.savemat(
    train_mat_path,
    mdict={
        'a': train_initial_vorticity,        # (Nb, s, s)
        'u': train_vorticity_evolution,      # (Nb, s, s, record_steps_train)
        't': time_vector_train  # (record_steps_train,)
    }
)
test_mat_path = os.path.join(out_dir, "ns_data_fno_test.mat")

scipy.io.savemat(
    test_mat_path,
    mdict={
        'a': test_initial_vorticity,        # (Nte, s, s)
        'u': test_vorticity_evolution,      # (Nte, s, s, record_steps_test)
        't': time_vector_test  # (record_steps_test,)
    }
)



import os
import imageio
import matplotlib.pyplot as plt
from matplotlib import cm

def save_field_images(field_series, output_dir, prefix, Nx, T):
    os.makedirs(output_dir, exist_ok=True)
    steps = field_series.shape[0]
    dt = T / (steps - 1)
    middle = steps // 2

    # три “статичных” кадра: начало, середина, конец
    indices = [0, middle, steps - 1]
    labels  = [f"t={i*dt:.2f}s" for i in indices]
    snapshots = []
    for idx, label in zip(indices, labels):
        fig, ax = plt.subplots()
        cp = ax.contourf(field_series[idx].reshape(Nx, Nx).T, cmap=cm.jet)
        fig.colorbar(cp, ax=ax)
        ax.set_title(f'{prefix} {label}')
        fname = os.path.join(output_dir, f'{prefix}_{label}.png'.replace('.', 'p').replace('=', '').replace('s','s'))
        fig.savefig(fname, bbox_inches='tight')
        plt.close(fig)
        snapshots.append(fname)

    # gif по всем кадрам
    gif_path = os.path.join(output_dir, f'{prefix}_evolution.gif')
    frames = []
    for i in range(steps):
        fig, ax = plt.subplots()
        cp = ax.contourf(field_series[i].reshape(Nx, Nx).T, cmap=cm.jet)
        ax.set_title(f'{prefix} t={i*dt:.2f}s')
        buf = os.path.join(output_dir, f'_tmp_{i:04d}.png')
        fig.savefig(buf, bbox_inches='tight')
        plt.close(fig)
        frames.append(imageio.v2.imread(buf))
        os.remove(buf)

    imageio.mimsave(gif_path, frames, fps=30)
    return snapshots, gif_path

# размеры
Nx = size   # у вас размер сетки
# длительности
T_tr = T_train
T_te = T_test

# индексы примеров для визуализации
i_tr = 0
i_te = 0

# фасуется папка для визуализации
vis_root = os.path.join("Data", filename, "visualisation")
os.makedirs(vis_root, exist_ok=True)


train_series = train_vorticity_evolution[i_tr].transpose(2, 0, 1)
snap_tr, gif_tr = save_field_images(
    train_series,
    os.path.join(vis_root, f"train_{i_tr:02d}"),
    f"train{i_tr:02d}",
    Nx=Nx,
    T=T_tr
)

test_series = test_vorticity_evolution[i_te].transpose(2, 0, 1)
snap_te, gif_te = save_field_images(
    test_series,
    os.path.join(vis_root, f"test_{i_te:02d}"),
    f"test{i_te:02d}",
    Nx=Nx,
    T=T_te
)

import numpy as np
import imageio
import matplotlib.pyplot as plt
from matplotlib import cm
import os

def save_speed_images(u_series, v_series, output_dir, prefix, Nx, T):
    """
    Визуализация поля скорости как скалярного поля |u| = sqrt(u^2 + v^2).
    Сохраняет снимки в начале, середине и конце, и GIF всей эволюции.
    """
    os.makedirs(output_dir, exist_ok=True)
    steps = u_series.shape[0]
    dt = T / (steps - 1)
    middle = steps // 2

    # вычисляем модуль скорости
    speed = np.sqrt(u_series**2 + v_series**2)  # shape (steps, Nx, Nx)

    indices = [0, middle, steps - 1]
    labels  = [f"t={i*dt:.2f}s" for i in indices]
    snapshots = []

    for idx, label in zip(indices, labels):
        fig, ax = plt.subplots(figsize=(4,4))
        cf = ax.contourf(speed[idx].T, cmap=cm.viridis)
        fig.colorbar(cf, ax=ax, label='|u|')
        ax.set_title(f'{prefix} {label}')
        fname = os.path.join(output_dir,
                             f'{prefix}_{label.replace(".", "p").replace("=", "")}.png')
        fig.savefig(fname, bbox_inches='tight')
        plt.close(fig)
        snapshots.append(fname)

    # GIF всей эволюции
    gif_path = os.path.join(output_dir, f'{prefix}_evolution.gif')
    frames = []
    for i in range(steps):
        fig, ax = plt.subplots(figsize=(4,4))
        cf = ax.contourf(speed[i].T, cmap=cm.viridis)
        ax.set_title(f'{prefix} t={i*dt:.2f}s')
        buf = os.path.join(output_dir, f'_tmp_speed_{i:04d}.png')
        fig.savefig(buf, bbox_inches='tight')
        plt.close(fig)
        frames.append(imageio.v2.imread(buf))
        os.remove(buf)

    imageio.mimsave(gif_path, frames, fps=30)
    return snapshots, gif_path


i_tr = 0
i_te = 0

# train_psi_y_evolution — это u_x (dψ/dy), train_psi_x_evolution — это u_y (−dψ/dx)
train_u = train_psi_y_evolution[i_tr].transpose(2, 0, 1)  # shape: (steps, Nx, Nx)
train_v = train_psi_x_evolution[i_tr].transpose(2, 0, 1)

speed_vis_root = os.path.join("Data", filename, "visualisation", f"train_speed_{i_tr:02d}")
snap_speed_tr, gif_speed_tr = save_speed_images(
    train_u, train_v, speed_vis_root,
    prefix=f"train_speed{i_tr:02d}",
    Nx=Nx, T=T_tr
)

import os
import numpy as np
import matplotlib.pyplot as plt

# индекс образца
i_sample = 0

# координаты точек
coords = [(0.5, 0.5), (0.8, 0.8), (0.2, 0.2)]

# параметры
Nx = size
t_vec = time_when_record_train[0].cpu().numpy()

# создаём папку для временных рядов
ts_dir = os.path.join("Data", filename, "visualisation", "timeseries")
os.makedirs(ts_dir, exist_ok=True)

# собираем данные
vort_ts = {}
speed_ts = {}
for x0, y0 in coords:
    ix = int(x0 * Nx)
    iy = int(y0 * Nx)

    ω = train_vorticity_evolution[i_sample, ix, iy, :]
    u_comp = train_psi_y_evolution[i_sample, ix, iy, :]
    v_comp = train_psi_x_evolution[i_sample, ix, iy, :]
    speed = np.sqrt(u_comp**2 + v_comp**2)

    vort_ts[(x0, y0)] = ω
    speed_ts[(x0, y0)] = speed

# 1) Сохранение графика завихренности ω(t)
fig, ax = plt.subplots(figsize=(6,4))
for (x0, y0), ω in vort_ts.items():
    ax.plot(t_vec, ω, label=f'ω at ({x0},{y0})')
ax.set_xlabel('t, s')
ax.set_ylabel('vorticity ω')
ax.set_title('Колебания завихренности в трёх точках')
ax.legend()
fig.tight_layout()
fig_path = os.path.join(ts_dir, "vorticity_timeseries.png")
fig.savefig(fig_path, dpi=150)
plt.close(fig)

# 2) Сохранение графика модуля скорости |u|(t)
fig, ax = plt.subplots(figsize=(6,4))
for (x0, y0), s in speed_ts.items():
    ax.plot(t_vec, s, label=f'|u| at ({x0},{y0})')
ax.set_xlabel('t, s')
ax.set_ylabel('speed |u|')
ax.set_title('Колебания модуля скорости в трёх точках')
ax.legend()
fig.tight_layout()
fig_path = os.path.join(ts_dir, "speed_timeseries.png")
fig.savefig(fig_path, dpi=150)
plt.close(fig)

print(f"Графики сохранены в: {ts_dir}")


Steps time: 100%|██████████| 15000/15000 [00:13<00:00, 1077.97it/s]


0 10 15.853180599398911


Steps time: 100%|██████████| 15000/15000 [00:18<00:00, 806.54it/s]


1 20 34.754797015339136


Steps time: 100%|██████████| 15000/15000 [00:13<00:00, 1085.80it/s]


...

2 30 85.6886853389442


Steps time: 100%|██████████| 25000/25000 [00:23<00:00, 1086.02it/s]


3 40 108.91084361355752


Steps time: 100%|██████████| 25000/25000 [00:28<00:00, 890.14it/s] 


4 50 137.20588806271553
Графики сохранены в: Data/DON_nu_0.0001_size_64_dtRecord_0.02_dtCycle_0.001_trainTime_15_testTime_25_NumTrain_200_NumTest_50/visualisation/timeseries
